## | `05_signal_analysis` | Tests whether promotions, holidays and weather move demand, with method checks (selection bias, event definitions, confounders) |

In [10]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

In [11]:
con = duckdb.connect(r'C:\Users\bhupe\demand-sensing-favorita\data\favorita.duckdb', read_only=True)

In [3]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect('../data/favorita.duckdb', read_only=True)   # read-only: can't change tables by mistake

promo = con.sql("""
SELECT family,
       is_on_promotion,
       AVG(unit_sales_clean) AS avg_units,
       COUNT(*)              AS n_rows
FROM demand_daily
WHERE date >= '2016-01-01'                -- sensing period
GROUP BY family, is_on_promotion
""").df()

lift = promo.pivot(index="family", columns="is_on_promotion", values="avg_units")
lift["promo_lift_pct"] = (lift[True] - lift[False]) / lift[False] * 100
lift

is_on_promotion,False,True,promo_lift_pct
family,,,
BEVERAGES,9.234514,24.067860,160.629415
DAIRY,6.216139,8.521249,37.082671
MEATS,9.910314,17.805367,79.665013


** **Finding:** Naive promo lift (2016+, avg units per item-store-day, promo vs non-promo): Beverages +161%, Meats +80%, Dairy +37%.

**Caveat:** this comparison overstates the true effect. (1) Selection bias: promoted items tend to be faster movers. (2) Filled zero-sales days are all flagged non-promo, lowering the non-promo average. (3) Promotions coincide with weekends and holidays. A fair estimate compares each item-store against its own non-promo baseline, then aggregates weighted by volume. The ML model will estimate promo effects controlling for item, store and calendar.

In [4]:
fair = con.sql("""
WITH base AS (
    SELECT family, item_nbr, store_nbr, is_on_promotion, unit_sales_clean
    FROM demand_daily
    WHERE date >= '2016-01-01'
      AND NOT is_holiday_national AND NOT is_holiday_local AND NOT is_event   -- remove holiday confounding
),
pair AS (                                                                     -- one row per item × store
    SELECT family, item_nbr, store_nbr,
           AVG(unit_sales_clean) FILTER (WHERE is_on_promotion)     AS avg_promo,
           AVG(unit_sales_clean) FILTER (WHERE NOT is_on_promotion) AS avg_no_promo,
           COUNT(*)              FILTER (WHERE is_on_promotion)     AS promo_days
    FROM base
    GROUP BY family, item_nbr, store_nbr
)
SELECT family,
       COUNT(*)        AS pairs_with_promo,
       SUM(promo_days) AS promo_days,
       SUM(promo_days * (avg_promo - avg_no_promo))
         / SUM(promo_days * avg_no_promo) * 100 AS fair_lift_pct          -- extra units ÷ baseline units on promo days
FROM pair
WHERE promo_days > 0 AND avg_no_promo > 0                                  -- needs both promo and non-promo days
GROUP BY family
ORDER BY family
""").df()

compare = fair.merge(lift[["promo_lift_pct"]].reset_index(), on="family", validate="one_to_one")
compare[["family", "pairs_with_promo", "promo_days", "promo_lift_pct", "fair_lift_pct"]]

,family,pairs_with_promo,promo_days,promo_lift_pct,fair_lift_pct
0,BEVERAGES,4049,157287.0,160.629415,78.799459
1,DAIRY,2035,142326.0,37.082671,40.855713
2,MEATS,470,36628.0,79.665013,57.228276


**Finding:** Within-item promo lift (each item-store vs its own non-promo baseline, holidays/events excluded, weighted by promo days): Beverages [x]%, Dairy [y]%, Meats [z]% vs naive [161 / 37 / 80]%. Beverages shrinks most, so its naive lift was driven largely by item mix (fast movers are promoted). Dairy changes least, possibly because perishables are promoted to clear stock rather than because they're fast movers (hypothesis).

**Limitations:** does not capture post-promo dips (pull-forward), cannibalization of other items, stockouts during promos, promo depth (no price data), or day-of-week mix. The ML model will estimate promo effects jointly with calendar features; pull-forward can be captured with lagged promo features.

In [5]:
holiday = con.sql("""
WITH store_day AS (                                          -- grain: store × family × day
    SELECT store_nbr, family, date,
           SUM(unit_sales_clean)      AS units,
           BOOL_OR(is_holiday_national) AS is_hol,
           BOOL_OR(is_event)           AS is_evt
    FROM demand_daily
    WHERE date >= '2016-01-01'
    GROUP BY store_nbr, family, date
),
typed AS (                                                   -- label each day
    SELECT *,
           CASE WHEN is_hol THEN 'holiday'
                WHEN LEAD(is_hol) OVER (PARTITION BY store_nbr, family ORDER BY date) THEN 'day_before_holiday'
                WHEN is_evt THEN 'event'
                ELSE 'normal' END AS day_type
    FROM store_day
),
store_avg AS (                                               -- grain: store × family × day type
    SELECT store_nbr, family, day_type, AVG(units) AS avg_units, COUNT(*) AS n_days
    FROM typed
    GROUP BY store_nbr, family, day_type
),
normal AS (
    SELECT store_nbr, family, avg_units AS normal_units
    FROM store_avg WHERE day_type = 'normal'
)
SELECT s.family, s.day_type,
       SUM(s.n_days) AS store_days,
       SUM(s.avg_units) / SUM(n.normal_units) * 100 - 100 AS lift_vs_normal_pct
FROM store_avg s
JOIN normal n USING (store_nbr, family)                       -- each store vs its own normal
WHERE s.day_type <> 'normal'
GROUP BY s.family, s.day_type
ORDER BY s.family, s.day_type
""").df()
holiday

,family,day_type,store_days,lift_vs_normal_pct
0,BEVERAGES,day_before_holiday,221.0,-6.706021
1,BEVERAGES,event,390.0,9.480334
2,BEVERAGES,holiday,351.0,10.361306
3,DAIRY,day_before_holiday,221.0,-8.967758
4,DAIRY,event,390.0,-1.497470
5,DAIRY,holiday,351.0,11.889009
6,MEATS,day_before_holiday,221.0,-5.354193
7,MEATS,event,390.0,-2.689792
8,MEATS,holiday,351.0,2.291116


In [6]:
holiday = con.sql("""
WITH store_day AS (                                          -- grain: store × family × day
    SELECT store_nbr, family, date,
           SUM(unit_sales_clean)      AS units,
           BOOL_OR(is_holiday_national) AS is_hol,
           BOOL_OR(is_event)           AS is_evt
    FROM demand_daily
    WHERE date >= '2016-01-01'
    GROUP BY store_nbr, family, date
),
typed AS (                                                   -- label each day
    SELECT *,
           CASE WHEN is_hol THEN 'holiday'
                WHEN LEAD(is_hol) OVER (PARTITION BY store_nbr, family ORDER BY date) THEN 'day_before_holiday'
                WHEN is_evt THEN 'event'
                ELSE 'normal' END AS day_type
    FROM store_day
),
store_avg AS (                                               -- grain: store × family × day type
    SELECT store_nbr, family, day_type, AVG(units) AS avg_units, COUNT(*) AS n_days
    FROM typed
    GROUP BY store_nbr, family, day_type
),
normal AS (
    SELECT store_nbr, family, avg_units AS normal_units
    FROM store_avg WHERE day_type = 'normal'
)
SELECT s.family, s.day_type,
       SUM(s.n_days) AS store_days,
       SUM(s.avg_units) / SUM(n.normal_units) * 100 - 100 AS lift_vs_normal_pct
FROM store_avg s
JOIN normal n USING (store_nbr, family)                       -- each store vs its own normal
WHERE s.day_type <> 'normal'
GROUP BY s.family, s.day_type
ORDER BY s.family, s.day_type
""").df()
holiday

,family,day_type,store_days,lift_vs_normal_pct
0,BEVERAGES,day_before_holiday,221.0,-6.706021
1,BEVERAGES,event,390.0,9.480334
2,BEVERAGES,holiday,351.0,10.361306
3,DAIRY,day_before_holiday,221.0,-8.967758
4,DAIRY,event,390.0,-1.497470
5,DAIRY,holiday,351.0,11.889009
6,MEATS,day_before_holiday,221.0,-5.354193
7,MEATS,event,390.0,-2.689792
8,MEATS,holiday,351.0,2.291116


**Finding:** Days flagged as national holidays show higher sales (Beverages +10%, Dairy +12%, Meats +2%), while the day before is lower (−5% to −9%). This contradicts the usual pre-holiday stock-up pattern and is likely a **definition artefact**: `is_holiday_national` includes "Additional" days (e.g. 21–24 Dec), which are peak shopping days, so the pre-Christmas rush counts as "holiday" and "day before" falls on ordinary days.

**Design decision:** for modelling, (1) group consecutive holiday days into one holiday period, (2) define day-before relative to the period start, (3) keep holiday type as a separate feature (Additional ≠ Holiday), and (4) add days-to/days-since holiday features. Effects are learned by the model per item rather than set as fixed uplift %.

In [7]:
weather_fx = con.sql("""
WITH store_day AS (                                                  -- grain: store × family × day
    SELECT store_nbr, family, date, city,
           SUM(unit_sales_clean)    AS units,
           ANY_VALUE(temp_max_deg_c)   AS temp_max,                   -- same for every item that day
           ANY_VALUE(precipitation_mm) AS rain_mm,
           BOOL_OR(is_holiday_national OR is_holiday_local OR is_event) AS is_special
    FROM demand_daily
    WHERE date >= '2016-01-01'
    GROUP BY store_nbr, family, date, city
),
banded AS (                                                          -- label days, normal days only
    SELECT *,
           NTILE(4) OVER (PARTITION BY store_nbr, family ORDER BY temp_max) AS temp_quartile,  -- 1 = coolest 25%, 4 = hottest 25%
           CASE WHEN rain_mm >= 5 THEN 'rainy' ELSE 'dry' END AS rain_band
    FROM store_day
    WHERE NOT is_special
),
store_fx AS (                                                        -- grain: store × family
    SELECT store_nbr, family,
           CASE WHEN city IN ('Guayaquil','Machala','Esmeraldas','Daule') THEN 'Costa' ELSE 'Sierra' END AS zone,
           AVG(units) FILTER (WHERE temp_quartile = 4)   AS hot_units,
           AVG(units) FILTER (WHERE temp_quartile = 1)   AS cool_units,
           AVG(units) FILTER (WHERE rain_band = 'rainy') AS rainy_units,
           AVG(units) FILTER (WHERE rain_band = 'dry')   AS dry_units,
           COUNT(*)   FILTER (WHERE rain_band = 'rainy') AS rainy_days
    FROM banded
    GROUP BY store_nbr, family, city
)
SELECT family, zone,
       SUM(hot_units) / SUM(cool_units) * 100 - 100                               AS hot_vs_cool_pct,
       SUM(rainy_units) FILTER (WHERE rainy_days > 0)
         / SUM(dry_units) FILTER (WHERE rainy_days > 0) * 100 - 100               AS rainy_vs_dry_pct,
       SUM(rainy_days)                                                            AS rainy_store_days
FROM store_fx
GROUP BY family, zone
ORDER BY family, zone
""").df()
weather_fx

,family,zone,hot_vs_cool_pct,rainy_vs_dry_pct,rainy_store_days
0,BEVERAGES,Costa,5.947013,-4.102384,1038.0
1,BEVERAGES,Sierra,0.168927,2.782425,1728.0
2,DAIRY,Costa,1.282335,-6.515223,1038.0
3,DAIRY,Sierra,-4.293415,2.991938,1728.0
4,MEATS,Costa,-1.772059,-2.525725,1038.0
5,MEATS,Sierra,-4.308077,1.900732,1728.0


# 05 – Signal Analysis

**Purpose:** Test which signals actually move demand before building models, and decide how each should be used.

**Input:** `demand_daily` (2016-01-01 → 2017-08-15). Aggregated in DuckDB, analysed in pandas.

**Key findings:**
| Signal | Result | Model use |
|---|---|---|
| Promotion | Naive lift Beverages +161%, Meats +80%, Dairy +37%, overstated by item mix; within-item lift is lower (Beverages shrinks most) | Plan + sensing (known in advance) |
| Holidays | Flagged holidays +2–12%, day before −5–9%. Likely a definition artefact: "Additional" days (e.g. 21–24 Dec) are peak shopping days counted as holidays | Plan + sensing, as holiday periods, holiday type and days-to/since features |
| Weather | Hot days: Beverages +6% in Costa, ~0% in Sierra. Rain: −3% to −7% in Costa, +2–3% in Sierra (likely seasonal/noise) | Sensing only; production would use weather forecasts |

**Method lessons:**
- Compare each item-store (or store) with **itself**; naive group comparisons mix in item and store differences.
- Check **event definitions** before trusting an effect.
- Seasonality and trend confound simple comparisons. Models need calendar features alongside signals.
- **Limitations:** no post-promo dip, cannibalization, stockouts or promo depth (no price data).

In [8]:
con.close()